In [1]:
import threading
import time
import random

class PhysicalAsset:
    """
    Represents a physical asset (e.g., a sensor, a machine).  In a real-world
    scenario, this class would interface with actual hardware.  For this
    simulation, we'll generate random data.
    """
    def __init__(self, asset_id, initial_temperature=25.0, initial_pressure=100.0):
        """
        Initializes the physical asset.

        Args:
            asset_id (str): Unique identifier for the asset.
            initial_temperature (float): Initial temperature of the asset.
            initial_pressure (float): Initial pressure of the asset.
        """
        self.asset_id = asset_id
        self.temperature = initial_temperature
        self.pressure = initial_pressure
        self.running = True  # Simulate asset running status

    def get_data(self):
        """
        Simulates getting data from sensors.  In a real application, this would
        involve reading sensor values.  Here, we add a small random variation
        to simulate real-world fluctuations and potential issues.
        """
        if not self.running:
            return {"temperature": None, "pressure": None, "status": "offline"}

        # Simulate normal operation with small fluctuations
        self.temperature += random.uniform(-0.5, 0.5)
        self.pressure += random.uniform(-1.0, 1.0)

        # Simulate occasional spikes or drops (potential issues)
        if random.random() < 0.05:  # 5% chance of a significant change
            self.temperature += random.uniform(-5, 5)
        if random.random() < 0.05:
            self.pressure += random.uniform(-10, 10)

        # Keep values within a reasonable range
        self.temperature = max(0, min(self.temperature, 100))  # Example range: 0-100
        self.pressure = max(50, min(self.pressure, 150))      # Example range: 50-150

        return {"temperature": self.temperature, "pressure": self.pressure, "status": "online"}

    def run(self):
        """Simulates the asset running and generating data."""
        self.running = True

    def stop(self):
        """Simulates the asset being stopped."""
        self.running = False

    def __repr__(self):
        return f"PhysicalAsset(id={self.asset_id}, temp={self.temperature:.2f}, pressure={self.pressure:.2f}, status={'Online' if self.running else 'Offline'})"


class DigitalTwin:
    """
    Represents the digital counterpart of a physical asset.  It receives data
    from the physical asset and maintains a virtual representation of its state.
    """
    def __init__(self, asset_id):
        """
        Initializes the digital twin.

        Args:
            asset_id (str): Unique identifier, matching the physical asset.
        """
        self.asset_id = asset_id
        self.temperature = None
        self.pressure = None
        self.status = "Unknown"
        self.data_history = []  # Store historical data for analysis

    def update_data(self, data):
        """
        Updates the digital twin's state with new data from the physical asset.

        Args:
            data (dict): A dictionary containing the asset's data
                           (e.g., {"temperature": 25.5, "pressure": 101.2}).
        """
        if data is None:
            self.status = "Offline"
            return  # Don't update None values

        self.temperature = data["temperature"]
        self.pressure = data["pressure"]
        self.status = data["status"]

        # Add timestamped data to history
        self.data_history.append({"timestamp": time.time(), "data": data})
        # Keep only the last 100 data points to prevent memory issues.  This
        # should be configurable in a real application.
        self.data_history = self.data_history[-100:]

    def get_status(self):
        """Returns the current status of the digital twin."""
        return self.status

    def get_data_history(self):
        """Returns the historical data."""
        return self.data_history

    def __repr__(self):
        return f"DigitalTwin(id={self.asset_id}, temp={self.temperature:.2f}, pressure={self.pressure:.2f}, status={self.status})"


def simulate_iot_data(physical_asset, digital_twin):
    """
    Simulates the process of an IoT device sending data to its digital twin.
    Runs in a separate thread to mimic real-time data updates.

    Args:
        physical_asset (PhysicalAsset): The physical asset to simulate.
        digital_twin (DigitalTwin): The digital twin to update.
    """
    while True:
        data = physical_asset.get_data()  # Get data from the physical asset
        digital_twin.update_data(data)      # Update the digital twin
        time.sleep(1)  # Simulate data transmission frequency (e.g., every 1 second)

def analyze_data(digital_twin):
    """
    Simulates analyzing data from the digital twin.  This could involve
    detecting anomalies, predicting failures, or optimizing performance.
    Runs in a separate thread.

    Args:
        digital_twin (DigitalTwin): The digital twin to analyze.
    """
    while True:
        # Example: Check for temperature anomalies
        if digital_twin.temperature is not None: #check if it has received any data
            if digital_twin.temperature > 80:
                print(f"Alert: High temperature detected for {digital_twin.asset_id}: {digital_twin.temperature:.2f}°C")
            elif digital_twin.temperature < 10:
                print(f"Alert: Low temperature detected for {digital_twin.asset_id}: {digital_twin.temperature:.2f}°C")

        # Example: Check for pressure anomalies
        if digital_twin.pressure is not None:
            if digital_twin.pressure > 140:
                print(f"Alert: High pressure detected for {digital_twin.asset_id}: {digital_twin.pressure:.2f} Pa")
            elif digital_twin.pressure < 60:
                print(f"Alert: Low pressure detected for {digital_twin.asset_id}: {digital_twin.pressure:.2f} Pa")

        if digital_twin.status == "Offline":
            print(f"Alert: {digital_twin.asset_id} is offline!")
        time.sleep(5)  # Check for anomalies every 5 seconds


if __name__ == "__main__":
    # Create a physical asset and its digital twin
    asset_1 = PhysicalAsset("Asset001")
    twin_1 = DigitalTwin("Asset001")

    # Start the data simulation in a separate thread
    iot_thread = threading.Thread(target=simulate_iot_data, args=(asset_1, twin_1,))
    iot_thread.daemon = True  # Allow the main thread to exit even if this thread is running
    iot_thread.start()

    # Start the data analysis in a separate thread
    analysis_thread = threading.Thread(target=analyze_data, args=(twin_1,))
    analysis_thread.daemon = True
    analysis_thread.start()

    # Simulate the asset running for a while
    print(f"Initial State: {asset_1}")
    print(f"Initial State: {twin_1}")
    time.sleep(10)  # Simulate the system running for 10 seconds

    # Simulate stopping the asset
    print("Stopping Asset001...")
    asset_1.stop()
    time.sleep(5)

    #Simulate Running the asset again
    print("Running Asset001 again...")
    asset_1.run()
    time.sleep(5)

    # Print the final state of the digital twin
    print(f"Final State: {twin_1}")

    # Print the data history
    print(f"Data History: {twin_1.get_data_history()}")

    print("Exiting...")


Initial State: PhysicalAsset(id=Asset001, temp=25.27, pressure=99.08, status=Online)
Initial State: DigitalTwin(id=Asset001, temp=25.27, pressure=99.08, status=online)
Stopping Asset001...
Running Asset001 again...
Final State: DigitalTwin(id=Asset001, temp=23.50, pressure=88.50, status=online)
Data History: [{'timestamp': 1745177269.6434288, 'data': {'temperature': 25.26590788464567, 'pressure': 99.08316914993586, 'status': 'online'}}, {'timestamp': 1745177270.6441364, 'data': {'temperature': 25.60248548813507, 'pressure': 98.27142520933741, 'status': 'online'}}, {'timestamp': 1745177271.6443105, 'data': {'temperature': 25.137726095207853, 'pressure': 98.9167681533325, 'status': 'online'}}, {'timestamp': 1745177272.6444983, 'data': {'temperature': 24.778812381559966, 'pressure': 99.07400779466475, 'status': 'online'}}, {'timestamp': 1745177273.64489, 'data': {'temperature': 24.86532595173675, 'pressure': 99.5009951499803, 'status': 'online'}}, {'timestamp': 1745177274.6450372, 'data':